# **Agent's tools**

## **Imports and environment configs**

In [28]:
from dotenv import load_dotenv

load_dotenv()

True

In [29]:
from openai import OpenAI

MODEL = "gpt-5.4-mini"

openai_client = OpenAI()

In [63]:
import json

from bibliometric_rag_assistant.ingestion.pubmed import search_pubmed
from bibliometric_rag_assistant.search_strategy import PUBMED_QUERY

## **Simple Agent - answers without tools**

In [39]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history
)

print(response.output_text)

Properly, an **agent** is something that:

1. **Acts on behalf of a goal or principal**
2. **Perceives some situation**
3. **Chooses actions**
4. **Can affect its environment**
5. Often does this **over time**, not just once

In the broadest sense, an agent is an entity that **selects actions in response to conditions** in order to achieve something.

### In different contexts

- **Everyday use:** a person or organization that represents someone else  
  - e.g. a real estate agent acts for a buyer or seller

- **Philosophy / cognitive science:** an entity with **intentional action**  
  - it has reasons, goals, and can be treated as a source of actions

- **Artificial intelligence:** a system that **perceives and acts in an environment** to maximize some objective  
  - e.g. a robot vacuum, a chess program, or a chatbot with tools

### A useful formal definition
In AI, an agent is often defined as:

> a system that maps percepts or observations to actions.

More fully, an **intelligent

In [40]:
history += response.output

history.append({"role": "user", "content": "What specifically is an AI agent?"})

second_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(second_response.output_text)

An **AI agent** is a system that:

- **perceives** its environment,
- **decides** what to do next,
- **takes actions**,
- and does so with some **goal-directed objective**.

In other words, it’s not just an AI model that produces outputs — it’s an AI system that can **operate over time** and **interact with the world**.

### Core pieces of an AI agent
An AI agent usually has some combination of:

- **Inputs/perception**: text, images, sensors, API results, memory
- **Policy or decision-making**: rules, planning, learned behavior, or an LLM
- **Actions**: sending messages, calling tools, moving a robot, updating files, etc.
- **Goal/objective**: complete a task, maximize reward, satisfy a user, etc.
- **State or memory**: information retained across steps

### Simple example
A chatbot that only replies when you ask it a question is mostly just a **model** or **tool**.

But a system that can:
- read a task,
- break it into steps,
- call a search tool,
- check results,
- revise its plan,


In [41]:
history += second_response.output

history.append({"role": "user", "content": "According with the last answers, will you become an agent when i provide tools and call you in a loop?"})

third_response = openai_client.responses.create(
    model=MODEL,
    input=history,
)

print(third_response.output_text)

Yes — **in that setup, I can function as part of an agent**.

More precisely:

- **The model itself** is not automatically an agent just because it exists.
- But if you **give it tools**, **feed it observations**, and **call it repeatedly in a control loop**, then the **overall system** becomes an agentic system.

### Why
An agent is usually the **whole loop**, not just the language model:
1. **Observe**: I get the current state, task, or tool results.
2. **Decide**: I choose what to do next.
3. **Act**: I output a tool call or response.
4. **Repeat**: the orchestrator sends back the new state.

That loop is what makes it agent-like.

### Important nuance
I do **not** become fully autonomous on my own.  
I only act within the boundaries of:
- the tools you expose,
- the loop you run,
- the instructions you give,
- and the information you pass back.

So the agent is really:
> **the system composed of the model + tools + loop + memory/state + objective**

### Short answer
**Yes, if you p

## **Defining a tool**

In [77]:
search_pubmed_tool = {
    "type": "function",
    "name": "search_pubmed",
    "description": "Search PubMed and return matching PMIDs and the total count. Use this when the user asks how many articles exist on a topic, or wants a list of PubMed IDs for a query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "PubMed search query, using PubMed's field-tag syntax (e.g. [Title/Abstract], [Mesh])."
            },
            "limit": {
                "type": "integer",
                "description": "Maximum number of PMIDs to return."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}


## **Sending the question with the tool**

Mesmo com a ferramenta, vai responder "na lata". Não precisa de decidir se chama o não a ferramenta, já que a pergunta não tem a menor relação com o que a ferramenta pode fazer. 

In [78]:
history = [
    {"role": "user", "content": "What is an agent, properly?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

message ResponseOutputMessage(id='msg_0e034d20906fd4b6006aad98f4b38887d28ad3805a2d4706fe', content=[ResponseOutputText(annotations=[], text='An **agent**, properly, is **something that acts on behalf of a goal or principle**.\n\nMore precisely, in the broadest useful sense, an agent is an entity that:\n\n1. **can perceive or receive information** about its situation,\n2. **can choose or determine actions** based on that information,\n3. **can act** in the world or a system, and\n4. **does so in a way that is attributable to it** rather than purely to outside forces.\n\n### In philosophy\nAn agent is a being with **agency**: the capacity to initiate actions, make choices, and be responsible for them. Humans are paradigmatic agents. Depending on the theory, some animals, organizations, and artificial systems may count as agents too.\n\n### In AI and computer science\nAn agent is usually a system that:\n- observes its environment,\n- decides what to do,\n- and acts to achieve some objecti

Agora a pergunta terá relação com a funcionalidade da ferramenta, então na primeira chamada o modelo aprende o `args` que deve ser usado numa eventual chamada da ferramenta, mas a chamada em si só ocorre na segunda interação com a api.

- **Primeira chamada: o modelo decide se usa ou não uma tool**

In [80]:
instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


history = [
    {"role": "developer", "content": instructions},
    {"role": "user", "content": "Which pmids returns from the query in effect for this pipeline?"}
]


response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=history,
    tools=[search_pubmed_tool]  # SCHEMA/JSON definition
)

for item in response.output:
    print(item.type, item)

function_call ResponseFunctionToolCall(arguments='{"query":"( \\"Temporomandibular Joint\\"[Mesh] OR \\"temporomandibular joint\\"[Title/Abstract] ) AND ( \\"Joint Prosthesis\\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \\"2000/01/01\\"[Date - Publication] : \\"2025/12/31\\"[Date - Publication] )","limit":100}', call_id='call_CsKsrlvVoyguXTvPNWvdNKj2', name='search_pubmed', type='function_call', id='fc_089a61b00e849097006aad9a71b3f487d2aff44359c8720070', async_=None, caller=None, namespace=None, status='completed')


- **Resultado da primeira chamada: `arguments`**

Não há uma regra fixa, o modelo pode retornar em `arguments` a pergunta do usuário reformulada ou simplesmente copiar um query literal contido em `instructions` - nesse caso temos o `PUBMED_QUERY`.

In [81]:
call = response.output[0]
args = json.loads(call.arguments)

results = search_pubmed(**args)
result_json = json.dumps(results, indent=2)

result_json

'{\n  "header": {\n    "type": "esearch",\n    "version": "0.3"\n  },\n  "esearchresult": {\n    "count": "100",\n    "retmax": "100",\n    "retstart": "0",\n    "idlist": [\n      "41328123",\n      "41076314",\n      "40725551",\n      "40393895",\n      "40335417",\n      "40153056",\n      "40090550",\n      "39756382",\n      "39490003",\n      "39487073",\n      "39344460",\n      "39344453",\n      "39299812",\n      "39256068",\n      "39232864",\n      "38743036",\n      "38685147",\n      "38071499",\n      "37847424",\n      "37246472",\n      "36602552",\n      "36540452",\n      "36494246",\n      "36201690",\n      "35806971",\n      "35727653",\n      "35245490",\n      "34809840",\n      "34802615",\n      "34503888",\n      "34238634",\n      "34112570",\n      "33552802",\n      "32997445",\n      "32977036",\n      "32917485",\n      "32887620",\n      "32866483",\n      "32610046",\n      "32569038",\n      "32439381",\n      "32328836",\n      "32285518",\n      "3

O objeto `function_call` inteiro (o `ResponseFunctionToolCall`) é o que entra em history, não só o texto ou os argumentos - é justamente esse objeto completo (com `.call_id`, `.name`, `.arguments`) que a API depois precisa ver de volta no histórico pra saber a quê o `function_call_output` que você manda em seguida está respondendo.

In [82]:
history += response.output

history.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})


response = openai_client.responses.create(
    model=MODEL,
    input=history,
    tools=[search_pubmed_tool],
)

response.output_text

'The query returned **100 PMIDs** in this pipeline run. The PMIDs are:\n\n41328123, 41076314, 40725551, 40393895, 40335417, 40153056, 40090550, 39756382, 39490003, 39487073, 39344460, 39344453, 39299812, 39256068, 39232864, 38743036, 38685147, 38071499, 37847424, 37246472, 36602552, 36540452, 36494246, 36201690, 35806971, 35727653, 35245490, 34809840, 34802615, 34503888, 34238634, 34112570, 33552802, 32997445, 32977036, 32917485, 32887620, 32866483, 32610046, 32569038, 32439381, 32328836, 32285518, 32035835, 31918888, 31594444, 31168422, 30965154, 30641280, 30615847, 30602388, 30316660, 29178982, 29077682, 28843402, 28711310, 28688869, 28687468, 28623686, 28412789, 28337986, 28161135, 29563484, 27717819, 27726907, 27487617, 27481673, 27134154, 26897300, 26889353, 26687154, 26611373, 25818493, 25483447, 25483442, 25102890, 25048029, 24861472, 24480620, 24035523, 23720535, 23618995, 22986256, 22885491, 22883323, 21470752, 21077388, 20822842, 20594807, 20508523, 19660913, 19251399, 188128

## **Agentic loop**

In [83]:
def make_call(call):
    # JSON string -> python dict
    args = json.loads(call.arguments)

    if call.name == "search_pubmed":
        result = search_pubmed(**args)

    # Turn results into JSON
    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        "call_id": call.call_id,
        "output": result_json,
    }

In [84]:
MAX_TOOL_CALLS = 2

instructions = f"You are helping analyze a PubMed literature review. The current search strategy's query is: {PUBMED_QUERY}"


def run_agent(question: str) -> dict:
    history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    call_log = []
    stop_reason = None
    tool_call_count = 0
    final_answer = None
    it = 0

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=MODEL,
            input=history,
            tools=[search_pubmed_tool],
        )

        history.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                if tool_call_count >= MAX_TOOL_CALLS:
                    stop_reason = 'max_tool_calls'
                    break

                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                tool_call_count += 1

                call_log.append({
                    'tool': item.name,
                    'arguments': json.loads(item.arguments),
                    'result_summary': call_output['output'][:200],
                })

                history.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                final_answer = item.content[0].text
                print('ASSISTANT:')
                print(final_answer)

        if stop_reason == 'max_tool_calls':
            print(f'Parando: limite de {MAX_TOOL_CALLS} chamadas de tool atingido.')
            break

        it += 1
        if not has_function_calls:
            stop_reason = 'end_turn'
            break

    return {
        'answer': final_answer,
        'call_log': call_log,
        'stop_reason': stop_reason,
    }

## **Testando com três perguntas conhecidas**

A terceira pergunta não tem relação nenhuma com PubMed — serve pra confirmar que o agente também sabe **não** chamar a tool quando ela não é necessária.

In [85]:
known_questions = [
    "How many PMIDs does the query in effect for this pipeline return?",
    "What is the first PMID returned by our current search?",
    "What is the capital of France?",
]

for question in known_questions:
    print("=" * 60)
    print("QUESTION:", question)
    result = run_agent(question)
    print("STOP REASON:", result["stop_reason"])
    print("CALL LOG:", result["call_log"])
    print("ANSWER:", result["answer"])
    print()

QUESTION: How many PMIDs does the query in effect for this pipeline return?
iteration #0...
function_call: search_pubmed {"query":"( \"Temporomandibular Joint\"[Mesh] OR \"temporomandibular joint\"[Title/Abstract] ) AND ( \"Joint Prosthesis\"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( \"2000/01/01\"[Date - Publication] : \"2025/12/31\"[Date - Publication] )","limit":20}
iteration #1...
ASSISTANT:
The query returns **100 PMIDs** in total.
STOP REASON: end_turn
CALL LOG: [{'tool': 'search_pubmed', 'arguments': {'query': '( "Temporomandibular Joint"[Mesh] OR "temporomandibular joint"[Title/Abstract] ) AND ( "Joint Prosthesis"[Mesh] OR prosthe*[Title/Abstract] OR replacement[Title/Abstract] OR alloplastic[Title/Abstract] ) AND ( infection*[Title/Abstract] OR antibiotic*[Title/Abstract] OR prophylaxis[Title/Abstract] ) AND ( "2000/01/0